# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook provides a guided workflow for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The underlying structure is described via a Croissant schema accessible via URL. All entities (record sets, fields, columns, etc.) are referenced by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset Croissant schema can be found at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display metadata main fields
meta = dataset.metadata
print(f"Title: {meta.name}")
print(f"Version: {meta.version}")
print(f"Identifier: {meta.identifier}")
print(f"Description: {meta.description}\n")
print(f"Temporal coverage: {meta.temporalCoverage}")
print(f"Spatial coverage: {meta.spatialCoverage}")
print(f"Published: {meta.datePublished}")
print(f"License: {meta.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets in the dataset by @id (Croissant v1.0 syntax)
record_sets = dataset.record_sets

print("Available record sets (by @id):")
for rs in record_sets:
    print(f"  @id: {rs['@id']} | name: {rs.get('name', '(no name)')}")

# For each record set, list its fields (by @id and name)
for rs in record_sets:
    print(f"\nFields in record set @id: {rs['@id']}:")
    for field in rs.get('field', []):
        # field can be a dict or @id string (Croissant fields)
        if isinstance(field, dict):
            f_id = field.get('@id')
            f_name = field.get('name', '(no name)')
        else:
            f_id = field
            f_name = '(no name)'
        print(f"    - @id: {f_id} | name: {f_name}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s from the overview.

In [ ]:
# Build list of record set @ids for extraction
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        # Each record is a dict keyed by field @id; mlcroissant delivers them as such
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from record set @id: {rs_id}")
        else:
            print(f"No records found for record set @id: {rs_id}")
    except Exception as e:
        print(f"Could not load records for record set @id: {rs_id} (error: {e})")

# Display columns for the first non-empty DataFrame
sample_rs = None
for rs_id, df in dataframes.items():
    if not df.empty:
        sample_rs = rs_id
        break
if sample_rs:
    print(f"\nColumns in DataFrame for record set @id {sample_rs}:")
    print(dataframes[sample_rs].columns.tolist())
    display(dataframes[sample_rs].head())
else:
    print("No record sets with data were loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filtering, normalization, and aggregation by key field.

In [ ]:
# Choose a record set with data
if not dataframes:
    raise ValueError("No data was loaded into DataFrames.")

rs_id = sample_rs  # Use first available
df = dataframes[rs_id]

# Show columns and choose a numeric field and a group field based on the available columns
print(f"Columns available in record set @id {rs_id}:")
print(list(df.columns))

# Suggest a numeric field for analysis (heuristic: field with numeric dtype or suggest)
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
# If not found, try to select a likely field by name
if not numeric_field:
    for col in df.columns:
        if 'log' in col.lower() or 'coeff' in col.lower() or 'value' in col.lower():
            numeric_field = col
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except:
                pass
            break

if numeric_field:
    print(f"\nUsing numeric field: {numeric_field}")
else:
    raise ValueError("Could not infer a numeric field from the available columns.")

# Try to suggest a group field (categorical field)
group_field = None
for col in df.columns:
    # Exclude numeric_field
    if col != numeric_field and (df[col].dtype == object or df[col].dtype.name == 'category'):
        n_unique = df[col].nunique(dropna=True)
        if 1 < n_unique < min(10, 0.5*len(df)):
            group_field = col
            break
if group_field:
    print(f"Grouping by field: {group_field}")
else:
    print("No suitable group field found; proceeding without grouping.")

# Outlier filtering for numeric_field
if numeric_field:
    threshold = df[numeric_field].mean() + df[numeric_field].std()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nRecords where {numeric_field} > {threshold:.2f} (outlier filtering):")
    display(filtered_df.head())

    # Add normalized version of the numeric_field
    filtered_df.loc[:, f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - df[numeric_field].mean()) / df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # If there is a group field, show group-wise means
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df)
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and its relation to the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Box plot by group if group_field exists
    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to explore and process the FAIR² dataset using the `mlcroissant` library. We loaded the metadata, examined the dataset's structure, extracted tabular data by record set and field `@id`, conducted basic EDA (filtering, normalization, grouping), and visualized numeric distributions. This approach is generalizable to other Croissant-structured datasets—refer to entity `@id`s to ensure precise data selection and reproducibility.

<sup>For more details on Croissant schemas and the mlcroissant toolkit, visit [mlcommons/croissant](https://github.com/mlcommons/croissant).</sup>